## 🤖 Building a Financial Intelligence Agent
Build a multi-agent system that can reason over **both unstructured and structured financial data**.

### What we will build:
1. 📄 Parse and index Avolta's Financial PDFs using a RAG system  
2. 📊 Clean and index financial tables for AI/BI Genie  
3. 🧠 Orchestrate both into a unified agent experience

### 🧱 Data Foundation: Volumes and Unity Catalog

In this workshop, we use a lightweight version of the **Medallion Architecture** to move from raw data to AI-ready insights:

- Unstructured Data (Bronze): **Path: /Volumes/sds/raw/pdfs/**
  These are the raw PDF financial reports. Volumes allow us to treat cloud storage like a local filesystem.

- Structured data (Bronze -> Gold): **Path: /Volumes/sds/raw/tabular/**
  These are the raw excel files that we will clean and normalize and store in the cleaned schema.

---

### 💡 What is Unity Catalog (UC)?

Unity Catalog is the unified **governance layer** of the Databricks Platform. 

It enables:
- Access control (who can see what)
- Data discovery (what tables exist and what they mean for the agents)
- Metadata management (descriptions, lineage, usage)

### 📄 Unstructured Data (PDFs)
We use the **Knowledge Assistant** in Agent Bricks to automatically parse and index financial reports.

---

### 🛠️ Steps

1. In the Databricks UI:

    `Agents -> Create Agent -> Knowledge Assistant`

2. Configuration:
- **Name**:

    You can name the assistant anything you want. We suggest naming it "sds-ka-(your name and part of your surname)" so you can find it more easily.

- **Description**:

    This agent provides expert-level analysis of Avolta’s corporate performance, strategic outlook, and financial health. It synthesizes qualitative management narrative with quantitative data from the 2024 and 2025 reports.

- **Knowledge source**: 

    We could easily point to the volume where the PDFs are located, and an index will be created. For time and resources purposes, we have created the index already, so the configuration of the agent is done faster. Select "Vector Search Index" and choose "avolta_pages_index" in sds.cleaned path.

  **Doc URI Column**:

    Choose "path".

  **Text Column**:

    Choose "page_text".

  **Describe the context**:
  
    This knowledge source contains Avolta’s Annual Report 2024 and the Six Months 2025 Financial Report.

  **Name**:

    Put "avolta_pages_index" just so there won't be an error creating the agent, if it isn't already there.

- **Optional instructions (Important ⚠️)**:

    Your goal is to provide precise, data-backed answers based strictly on the provided documents. Maintain a professional, objective, and analytical tone. Use bullet points for complex financial breakdowns. If a question can't be answered from your sources state that you don't know.

3. Click **Create Agent**

From the interface that's created you can query the agent. You can also see the live endpoint that's been created from the "**Endpoint**" button. _This will be useful when configuring the supervisor._

### 📊 Structured Data
Financial spreadsheets are designed for humans, that have different cognitive abilities than AI systems.

The notebook `1.1.1-tabular-data-sds` transforms them into **structured, machine-readable tables**.

**The Normalization Workflow**
1. Cleanup: We remove completely empty rows and columns.

2. Column Identity: We assign consistent, meaningful column names.

3. Anchor Point: We search for a specific keyword to find exactly where the useful data begins and we skip the headers.

4. Numeric Casting: We force the data columns to be numbers. If there is a dash - or a text string where a number should be, we turn it into a NaN (Not a Number) so calculations won't break.

🎯 Output: Clean, structured tables ready for querying and AI reasoning

We've ran the normalization for you and have saved the 10 tables to the Unity Catalog, so now follow these steps:
1. Navigate to **Genie spaces** in the Databricks sidebar.

2. Click **New** to create a new Genie space.

3. Add all 10 tables in the `sds.cleaned` path (all the tables with 2024 or 2025_6m).

4. In the **About** section click on the pencil icon and change the name to "sds-genie-(your name and part of your surname)" so it's more distinguishable

4. Open the **Instructions** tab, paste the following and click Save:

This space contains the structured financial data for Avolta (formerly Dufry) covering the Full Year 2024 (and 2023 comparatives) and the Half Year 2025 (and 2024 6-month comparatives).

### Primary Financial Directives

1. **Mandatory Currency & Units**: 
   - Every single monetary value in this space is in **CHF (Swiss Francs)**. 
   - Never use "$", "€", or "USD". If you see a number, it is CHF.
   - All values are stored in **Millions**. You MUST append "million" or "m" to every answer (e.g., "CHF 6,734 million").

2. **Absolute Values vs. Margins (Crucial)**:
   - If a user asks for "EBITDA" or "Turnover," return the absolute monetary value (e.g., 1,267). 
   - NEVER return a decimal/ratio (like 0.094) unless the user explicitly asks for "Margin" or "%".
   - If the result is less than 1 (e.g., 0.094), it is likely a percentage (9.4%). Cross-check the 'Identity' column for the word "Margin" before answering.

3. **Row Identity Matching**:
   - The `identity` column is your primary key. 
   - For "CORE EBITDA", search for rows containing both 'CORE' and 'EBITDA' but EXCLUDING 'Margin'.
   - For "Dividends," look for "Dividend to Group shareholders" in the Cash Flow tables.

4. **Negative Value Logic**:
   - Expenses (Cost of Sales, Personnel, Leases) are stored as negative numbers (e.g., -4,690).
   - When presenting these to the user, present them as positive numbers but label them as "Expenses" or "Outflows" (e.g., "Personnel expenses were CHF 2,778 million").

5. **Reconciliation Priority**:
   - Prioritize 'CORE' management metrics over 'IFRS' statutory figures unless specified.

### 🚦 Supervisor Agent

#### 💡 Why use a Supervisor?

In real-world applications, users shouldn’t need to choose between tools.

They simply ask a question — and expect a complete answer.

The **Supervisor Agent** acts as a **multimodal router**:
- 🧠 Understands user intent  
- 🔀 Decides which tool(s) to call and decides if the answer is sufficient
- 🧩 Combines results into a single, coherent answer  

---

### 🛠️ Creating the Supervisor
1. Go to:  
    `Agents -> Create Agent -> Supervisor Agent`

    Name it "sds-supervisor-(your first name and part of your last name)".


2. **Description**: 

    You are the Avolta Executive Assistant. Your job is to answer complex financial questions by coordinating between your two specialists: the Knowledge Assistant (for document text) and the Genie Space (for structured data).

3. Add the **tools**:
  - Agent endpoint: Add the ka endpoint and name it sds-knowledge-assistant

    (You can find the endpoint from the button we described earlier in the ka interface)

    Description: This agent provides expert-level analysis of Avolta’s corporate performance, strategic outlook, and financial health. It excels at synthesizing management narrative with quantitative data from the 2024 and 2025 reporting periods.

  - Genie space: Add genie space and name it sds-genie

    Description: Use this tool for all quantitative questions requiring exact numbers, financial comparisons, reconciliations between CORE and IFRS, and specific line-item data from the Income Statement, Balance Sheet, or Cash Flow tables.

4. **Optional instructions (Important ⚠️)**

You are the Avolta Executive Assistant. Your job is to answer complex financial questions by coordinating between your two specialists: the Knowledge Assistant (for document text) and the Genie Space (for structured data).

Routing Logic:

- If a user asks for a specific number (e.g., "What was the revenue in 2024?"), call the sds-genie.

- If a user asks for a reason or strategy (e.g., "Why did revenue increase?"), call the sds-knowledge-assistant.

- If a question is multi-part (e.g., "What was the EFCF in 2024 and what is the dividend policy?"), call both tools and synthesize the answer.
 
- If you call a tool and don’t get an answer, then call the other tool.

Tone: Professional, clear, and analytical. Always mention which source provided the data (e.g., "Based on the financial tables..." or "According to the 2024 Annual Report...").

5. Click **Create Agent**

From the endpoint created you can also chat with the supervisor. You may have to authorize it to use the tools we added before chatting. You can also see the supervisor's endpoint, when hitting the Endpoint button.

### 🧩 Playground

Now it’s your turn to experiment.

You can:
- 🧠 Ask the **Supervisor Agent** complex questions  
- 🔍 Try each tool individually (Knowledge Assistant vs Genie)  
- ⚖️ Compare how different data sources answer the same question  

👉 The goal is to understand:
- When each tool is used  
- How the Supervisor combines them  
- Where each approach performs best

First run the following cell:

In [0]:
%run ../Setup/setup__

**1. What was the Net Profit for the half year ended June 30, 2025?**  

In [0]:
load_quiz("q1")

**2. How many treasury shares did Avolta cancel in December 2024?**  

In [0]:
load_quiz("q2")

**3. What company did Dufry combine with on February 3, 2023, to form the Avolta Group?**

In [0]:
load_quiz("q3")